# ENSEMBLE STRATEGY: Average XGBoost Predictions from Multiple Random Seeds
Train 3 XGBoost models with different seeds, average predictions to reduce variance

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("ENSEMBLE: XGBoost with Multiple Seeds")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/5] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

# Compress dtypes
def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== FEATURE ENGINEERING ==============
print("\n[2/5] Feature engineering (keep engineered features - they worked best)...")
def engineer_features(X):
    X_feat = X.copy()
    X_feat['row_mean'] = X.mean(axis=1)
    X_feat['row_std'] = X.std(axis=1)
    X_feat['row_median'] = X.median(axis=1)
    X_feat['row_skew'] = X.skew(axis=1)
    X_feat['row_max'] = X.max(axis=1)
    X_feat['row_min'] = X.min(axis=1)
    X_feat['row_max_min_diff'] = X_feat['row_max'] - X_feat['row_min']
    lag_cols = [c for c in X.columns if 'LagT' in c]
    if len(lag_cols) > 0:
        lag_subset = X[lag_cols]
        X_feat['lag_mean'] = lag_subset.mean(axis=1)
        X_feat['lag_std'] = lag_subset.std(axis=1)
    return X_feat

X_train_feat = engineer_features(X_train)
X_test_feat = engineer_features(X_test)
print(f"✓ Features: {X_train.shape[1]} → {X_train_feat.shape[1]}")

# ============== TRAIN ENSEMBLE WITH MULTIPLE SEEDS ==============
print("\n[3/5] Training 3 XGBoost models with different random seeds...")

seeds = [42, 123, 456]
ensemble_preds = np.zeros(len(X_test_feat))
all_scores = []

xgb_params = {
    'objective': 'reg:squarederror', 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.7, 'lambda': 1.0, 'alpha': 0.5,
    'n_jobs': 4, 'verbosity': 0
}

for seed_idx, seed in enumerate(seeds, 1):
    print(f"\n  Model {seed_idx}/3 (seed={seed}):")
    
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_indices = list(kf.split(X_train_feat))
    
    test_preds = np.zeros(len(X_test_feat))
    fold_scores = []
    
    for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
        X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = xgb.XGBRegressor(**xgb_params, n_estimators=800, random_state=seed)
        model.fit(X_tr, y_tr)
        
        test_preds += model.predict(X_test_feat) / 5
        r2 = r2_score(y_val, model.predict(X_val))
        fold_scores.append(r2)
        
        del model, X_tr, X_val, y_tr, y_val
        gc.collect()
    
    avg_r2 = np.mean(fold_scores)
    all_scores.append(avg_r2)
    ensemble_preds += test_preds / 3  # Average across 3 models
    print(f"    Avg R² (5-fold): {avg_r2:.6f}")

print(f"\n✓ Ensemble trained (3 models averaged)")
print(f"✓ Individual model scores: {[f'{s:.6f}' for s in all_scores]}")
print(f"✓ Ensemble Avg R²: {np.mean(all_scores):.6f}")

# ============== CREATE SUBMISSION ==============
print("\n[4/5] Creating submission...")
submission = pd.DataFrame({'ID': test_ids, 'TARGET': ensemble_preds})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

# ============== SAVE SUBMISSION ==============
print("\n[5/5] Saving submission...")
submission.to_csv('submission_ensemble.csv', index=False)
print(f"✓ Saved: submission_ensemble.csv")

print("\n" + "="*70)
print("ENSEMBLE SUBMISSION - READY FOR KAGGLE")
print("="*70)
print(f"✓ Strategy: 3 XGBoost models (seeds: {seeds})")
print(f"✓ Predictions averaged to reduce variance")
print(f"✓ Expected benefit: Smoother predictions → better generalization")
print("="*70)